**RF Model 1 (maxDepth = 10): Test F1 ~0.36**

In [22]:
# train/ test split
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.cache()
test_df = test_df.cache()
print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")

# inverse-frequency class weighting to deal with class imbalance
n_train = train_df.count()
n_classes = train_df.select("genre_index").distinct().count()

class_counts_df = (train_df.groupBy("genre_index").count().withColumn("class_weight",
        F.lit(float(n_train))/(F.lit(float(n_classes)) * F.col("count").cast("double"))
    ).select("genre_index", "class_weight"))

train_weighted = (train_df.join(F.broadcast(class_counts_df), on="genre_index", how="left").cache())

# build model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="genre_index",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=10,
    maxBins=128,
    seed=42,
)

rf_model = rf.fit(train_weighted)

# make predictions
train_preds = rf_model.transform(train_weighted)
test_preds = rf_model.transform(test_df)

# calculate metrics
acc_evaluator = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="f1")
train_acc = acc_evaluator.evaluate(train_preds)
test_acc = acc_evaluator.evaluate(test_preds)
train_f1 = f1_evaluator.evaluate(train_preds)
test_f1 = f1_evaluator.evaluate(test_preds)

print(f"\n{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'Accuracy':<20} {train_acc:>10.4f} {test_acc:>10.4f}")
print(f"{'Weighted F1':<20} {train_f1:>10.4f} {test_f1:>10.4f}")

# display per-class F1 scores
label_list = pipeline_model.stages[0].labels
per_class_evaluator = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="fMeasureByLabel")
print(f"\n{'Genre Bucket':<40} {'Test F1':>10}")
print("-" * 52)
for idx, label in enumerate(label_list):
    per_class_evaluator.setMetricLabel(float(idx))
    f1_val = per_class_evaluator.evaluate(test_preds)
    print(f"{label:<40} {f1_val:>10.4f}")

# display sample predictions with peek
print("\nSample predictions:")
peek(test_preds.select("release_name", "genre_index", "prediction"))

Train rows: 750,745
Test rows: 187,565

Metric                    Train       Test
------------------------------------------
Accuracy                 0.3693     0.3677
Weighted F1              0.3730     0.3716

Genre Bucket                                Test F1
----------------------------------------------------
Rock                                         0.2034
Electronic / Dance                           0.5911
Pop                                          0.1252
Jazz                                         0.6285
Classical / Orchestral / Opera               0.3545
Hip-Hop / Rap                                0.5485
Metal                                        0.5436
Experimental / Avant-garde / Noise           0.3020
Punk / Hardcore / Emo                        0.2844
Folk / Singer-Songwriter                     0.3316
Ambient / New Age                            0.3032
Country / Americana / Bluegrass              0.4470
R&B / Soul / Funk                            0.3914
Blues 

,release_name,genre_index,prediction
0,!Bailando!,14.0,16.0
1,"""180""",0.0,8.0
2,"""A divina comédia ou ando meio desligado""",0.0,8.0
3,"""Bingo Bango"" Remixes Plus ""Jus Tonight""",1.0,1.0
4,"""From a Capsule Underground""",0.0,4.0
5,"""I Wonder Who the Real Cannibals Are"" / There ...",7.0,6.0
6,"""Little Jazz"" Jazz",3.0,3.0
7,"""Pee Wee"" & ""Fingers""",3.0,3.0
8,"""Time Remembered""",3.0,3.0
9,"""Un Altro, bitte!""",7.0,3.0


**RF Model 2 (maxDepth = 15): Test F1 ~0.38**

In [24]:
# train/ test split
train_df2, test_df2 = final_df.randomSplit([0.8, 0.2], seed=42)
train_df2 = train_df2.cache()
test_df2  = test_df2.cache()
print(f"Train rows: {train_df2.count():,}")
print(f"Test rows: {test_df2.count():,}")

# inverse-frequency class weighting to deal with class imbalance
n_train2   = train_df2.count()
n_classes2 = train_df2.select("genre_index").distinct().count()

class_counts_df2 = (train_df2.groupBy("genre_index").count().withColumn("class_weight",
        F.lit(float(n_train2)) / (F.lit(float(n_classes2)) * F.col("count").cast("double"))
    ).select("genre_index", "class_weight"))

train_weighted2 = (train_df2.join(F.broadcast(class_counts_df2), on="genre_index", how="left").cache())

# build model
rf2 = RandomForestClassifier(
    featuresCol="features",
    labelCol="genre_index",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=12,
    maxBins=128,
    seed=42,
)

rf_model2 = rf2.fit(train_weighted2)

# make predictions
train_preds2 = rf_model2.transform(train_weighted2)
test_preds2  = rf_model2.transform(test_df2)

# calculate metrics
acc_evaluator2 = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="accuracy")
f1_evaluator2 = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="f1")
train_acc2 = acc_evaluator2.evaluate(train_preds2)
test_acc2  = acc_evaluator2.evaluate(test_preds2)
train_f1_2 = f1_evaluator2.evaluate(train_preds2)
test_f1_2  = f1_evaluator2.evaluate(test_preds2)

print(f"\n{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'Accuracy':<20} {train_acc2:>10.4f} {test_acc2:>10.4f}")
print(f"{'Weighted F1':<20} {train_f1_2:>10.4f} {test_f1_2:>10.4f}")

# display per-class F1 scores
label_list2 = pipeline_model.stages[0].labels
per_class_evaluator2 = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="fMeasureByLabel")
print(f"\n{'Genre Bucket':<40} {'Test F1':>10}")
print("-" * 52)
for idx, label in enumerate(label_list2):
    per_class_evaluator2.setMetricLabel(float(idx))
    f1_val2 = per_class_evaluator2.evaluate(test_preds2)
    print(f"{label:<40} {f1_val2:>10.4f}")

# display sample predictions with peek
print("\nSample predictions:")
peek(test_preds2.select("release_name", "genre_index", "prediction"))

Train rows: 750,745
Test rows: 187,565

Metric                    Train       Test
------------------------------------------
Accuracy                 0.3766     0.3737
Weighted F1              0.3789     0.3754

Genre Bucket                                Test F1
----------------------------------------------------
Rock                                         0.1962
Electronic / Dance                           0.5933
Pop                                          0.1255
Jazz                                         0.3728
Classical / Orchestral / Opera               0.8064
Hip-Hop / Rap                                0.5455
Metal                                        0.5473
Experimental / Avant-garde / Noise           0.3050
Punk / Hardcore / Emo                        0.2821
Folk / Singer-Songwriter                     0.3316
Ambient / New Age                            0.3027
Country / Americana / Bluegrass              0.4399
R&B / Soul / Funk                            0.4036
Blues 

,release_name,genre_index,prediction
0,!Bailando!,14.0,18.0
1,"""180""",0.0,8.0
2,"""A divina comédia ou ando meio desligado""",0.0,8.0
3,"""Bingo Bango"" Remixes Plus ""Jus Tonight""",1.0,1.0
4,"""From a Capsule Underground""",0.0,3.0
5,"""I Wonder Who the Real Cannibals Are"" / There ...",7.0,6.0
6,"""Little Jazz"" Jazz",3.0,3.0
7,"""Pee Wee"" & ""Fingers""",3.0,3.0
8,"""Time Remembered""",3.0,3.0
9,"""Un Altro, bitte!""",7.0,3.0


### 3. Fitting Analysis

We created two Random Forest models with different hyperparameters to evaluate how model complexity affected performance. Our first model used the following parameters:

- Trees = 50
- Max Depth = 10
- Max Bins = 128

This model achieved an F1 score of approximately 0.36 on the test dataset.

Our second model increased the maximum tree depth while keeping the other parameters constant:

- Trees = 50
- Max Depth = 12
- Max Bins = 128

This model achieved a slightly improved F1 score of approximately 0.38.

The improvement from increasing tree depth suggests that the original model was underfitting the data, since deeper trees were able to capture more complex relationships between the metadata and tag based features. Both models still appear to fall on the underfitting side of the fitting curve despite the improved performance. This is due to the noisy genre labels, sparse high-dimensional tag vectors, and the broad variability present in the MusicBrainz dataset whiah all contribute to the difficulty of this task.

We also experimented with additional Random Forest configurations using larger numbers of trees and greater depths. These configurations became computationally expensive because of the scale of the dataset and the size of the generated feature vectors.

The second Random Forest model performed best because the increased tree depth allowed the model to learn more detailed decision boundaries and better capture interactions between categorical metadata and tag derived features.

For Milestone 4, we plan to explore dimensionality reduction techniques such as PCA and SVD. Since the CountVectorizer step produces large sparse feature vectors, reducing dimensionality may help decrease computational cost while preserving important semantic structure within the tags. We also plan to explore clustering approaches such as K-Means in order to identify latent groupings between genres, artists, or releases that may not be captured directly through supervised classification. In addition, we may experiment with other distributed models such as Gradient Boosted Trees or XGBoost to evaluate whether boosting methods can better model the complex nonlinear relationships present in the data.


### 4. Conclusion Section

Our first model demonstrated that Random Forest classifiers can learn meaningful patterns from the MusicBrainz dataset and perform multiclass genre prediction at scale. The model achieved moderate performance, with an F1 score of approximately 0.38 across the 19 genre categories. While this indicates that the model was able to capture relationships between artist metadata, labels, geographic information, and tags, the task itself was challenging due to noisy and highly variable genre labels derived from user generated tags.

We experimented with adjusting hyperparameters, like number of trees and tree depth, in order to improve performance. However, increasing model complexity significantly increased computational cost and runtime because of the large scale of the dataset. A possible direction for improvement could be additional feature engineering and feature selection. Some tables in MusicBrainz contain rich numerical relationship data, while others contain sparse or noisy text metadata, so refining which features to include could improve performance without dramatically increasing computational requirements. Additional improvements could also come from improved genre normalization, balancing underrepresented genres, or experimenting with more advanced distributed models like Gradient Boosted Trees or XGBoost.

Distributed computing was essential for this task because the dataset contained millions of rows and high dimensional feature vectors generated from release tags. Spark allowed us to distribute preprocessing, joins, aggregations, and model training across multiple executors rather than relying on a single machine. Distributed preprocessing techniques like CountVectorizer, StringIndexer, and VectorAssembler enabled us to efficiently transform large categorical and text-based datasets into feature vectors suitable for machine learning. Without distributed computing, processing and training on a dataset of this scale would have been nearly impossible.